#### Imports

In [1]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import gc
import re
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from collections import defaultdict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    set_seed,
)


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


#### Configs

In [2]:
RND = 42
set_seed(RND)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

BATCH_SIZE = 256
MAX_LEN = 128
CONFIDENCE_THRESHOLD = 0.65

# !unzip -q "/content/drive/MyDrive/Colab Notebooks/Cleaned_output.zip" -d "/content/drive/MyDrive/Colab Notebooks"

COMMENTS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Cleaned_output/digikala-comments_parts")
OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Cleaned_output/Comments_ABSA")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SENTIMENT_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/pars_absa_sentiment_model"
ASPECT_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/pars_absa_aspect_model"

SENTIMENT_ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
BIO_ID2LABEL = {0: "O", 1: "B-ASP", 2: "I-ASP"}


device: cuda
GPU: Tesla T4


#### Load models & Tokenizer

In [3]:
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_PATH, use_fast=True)

print("Loading sentiment model...")
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_PATH).to(device)
sentiment_model.eval()

print("Loading aspect model...")
aspect_model = AutoModelForTokenClassification.from_pretrained(ASPECT_MODEL_PATH).to(device)
aspect_model.eval()

if getattr(sentiment_model.config, "id2label", None):
    SENTIMENT_ID2LABEL = {int(k): v for k, v in sentiment_model.config.id2label.items()}
if getattr(aspect_model.config, "id2label", None):
    BIO_ID2LABEL = {int(k): v for k, v in aspect_model.config.id2label.items()}


Loading tokenizer...
Loading sentiment model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading aspect model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

#### High-Performance Inference Helpers

In [4]:
@torch.inference_mode()
def predict_sentiment_batch(texts_or_pairs):
    if not texts_or_pairs:
        return []

    # Dynamic Padding برای سرعت بالا
    with torch.cuda.amp.autocast(enabled=(device == "cuda")):
        encoded = tokenizer(
            texts_or_pairs,
            padding=True,          # پدینگ پویا به اندازه بلندترین جمله بچ
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).to(device)

        logits = sentiment_model(**encoded).logits
        preds = logits.argmax(dim=1).cpu().numpy()

    return [SENTIMENT_ID2LABEL[int(p)] for p in preds]


@torch.inference_mode()
def predict_aspects_batch(texts):
    with torch.cuda.amp.autocast(enabled=(device == "cuda")):
        encoded = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_offsets_mapping=True,
            return_tensors="pt",
        )
        offsets = encoded.pop("offset_mapping").cpu().numpy()
        encoded = {k: v.to(device) for k, v in encoded.items()}

        logits = aspect_model(**encoded).logits
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(-1).cpu().numpy()
        max_probs = probs.max(-1).values.cpu().numpy()

    return preds, max_probs, offsets


def clean_aspect(span: str) -> str:
    span = span.strip()
    span = re.sub(r"\s+", " ", span)
    span = re.sub(r"^[^\w\s]+|[^\w\s]+$", "", span, flags=re.UNICODE)
    return span.strip()


def is_valid_aspect(span: str) -> bool:
    if not span or len(span) < 2:
        return False
    if not re.search(r"[\u0600-\u06FFa-zA-Z]", span):
        return False

    stop_words = {
        "ش", "رو", "که", "از", "به", "با", "این", "هم", "را", "و",
        "در", "می", "های", "ها", "یک", "همه", "برای", "ولی", "اما",
        "نیست", "هست", "شد", "میشه", "کنه", "میکنه", "خیلی", "اصلا", "انگار", "r"
    }
    if span in stop_words:
        return False
    return True


def decode_aspects(text, pred_ids, max_probs, offsets_row):
    aspects = []
    cur_start, prev_end = None, None
    current_probs = []

    for pred, prob, (start, end) in zip(pred_ids, max_probs, offsets_row):
        start, end = int(start), int(end)
        if start == 0 and end == 0:
            continue

        label = BIO_ID2LABEL.get(int(pred), "O")

        if label.startswith("B"):
            if cur_start is not None and prev_end is not None:
                span = clean_aspect(text[cur_start:prev_end])
                avg_p = sum(current_probs) / len(current_probs) if current_probs else 0
                if is_valid_aspect(span) and avg_p >= CONFIDENCE_THRESHOLD:
                    aspects.append(span)
            cur_start, prev_end = start, end
            current_probs = [prob]

        elif label.startswith("I") and cur_start is not None:
            prev_end = end
            current_probs.append(prob)

        else:
            if cur_start is not None and prev_end is not None:
                span = clean_aspect(text[cur_start:prev_end])
                avg_p = sum(current_probs) / len(current_probs) if current_probs else 0
                if is_valid_aspect(span) and avg_p >= CONFIDENCE_THRESHOLD:
                    aspects.append(span)
            cur_start, prev_end = None, None
            current_probs = []

    if cur_start is not None and prev_end is not None:
        span = clean_aspect(text[cur_start:prev_end])
        avg_p = sum(current_probs) / len(current_probs) if current_probs else 0
        if is_valid_aspect(span) and avg_p >= CONFIDENCE_THRESHOLD:
            aspects.append(span)

    seen, out = set(), []
    for a in aspects:
        if a not in seen:
            seen.add(a)
            out.append(a)
    return out


#### Main Loop over Files

In [5]:
parquet_files = sorted(COMMENTS_DIR.glob("*.parquet"))
print(f"\nFound {len(parquet_files)} raw parts")
print(f"Output Directory: {OUTPUT_DIR}\n")

aspect_stats_global = defaultdict(lambda: {"positive": 0, "negative": 0, "neutral": 0, "total": 0})

for file in parquet_files:
    output_path = OUTPUT_DIR / file.name

    if output_path.exists():
        print(f"Skipping existing file: {file.name}")
        try:
            df_ex = pd.read_parquet(output_path, columns=["predicted_aspects"])
            for row in df_ex["predicted_aspects"]:
                if row is None or (isinstance(row, float) and np.isnan(row)):
                    continue

                for item in row:
                    if isinstance(item, dict) and "aspect" in item and "sentiment" in item:
                        asp, sent = item["aspect"], item["sentiment"]
                        aspect_stats_global[asp][sent] += 1
                        aspect_stats_global[asp]["total"] += 1

            continue

        except Exception as e:
            print(f"⚠️ Warning: Could not read existing file {file.name}. Re-processing... Error: {e}")
            output_path.unlink(missing_ok=True)


    print("=" * 70)
    print("Processing:", file.name)

    df = pd.read_parquet(file)
    texts = df["raw_text_normalized"].fillna("").astype(str).tolist()
    n = len(texts)

    sorted_indices = sorted(range(n), key=lambda k: len(texts[k]))
    sorted_texts = [texts[i] for i in sorted_indices]

    all_sentiments_sorted = []
    all_aspect_pairs_sorted = []

    for i in tqdm(range(0, n, BATCH_SIZE), desc=file.name):
        batch = sorted_texts[i : i + BATCH_SIZE]

        # 1) Document Sentiment
        doc_sents = predict_sentiment_batch(batch)

        # 2) Aspect Extraction
        asp_preds, asp_probs, asp_offsets = predict_aspects_batch(batch)
        batch_aspects = [
            decode_aspects(text, asp_preds[j], asp_probs[j], asp_offsets[j])
            for j, text in enumerate(batch)
        ]

        # 3) Aspect Sentiment
        pair_inputs = []
        owners = []
        for j, (text, aspects) in enumerate(zip(batch, batch_aspects)):
            for a in aspects:
                pair_inputs.append((text, a))
                owners.append((j, a))

        flat_labels = []
        if pair_inputs:
            for q_i in range(0, len(pair_inputs), BATCH_SIZE):
                q_batch = pair_inputs[q_i : q_i + BATCH_SIZE]
                flat_labels.extend(predict_sentiment_batch(q_batch))

        pairs_per_row = [[] for _ in batch]
        for (j, a), lab in zip(owners, flat_labels):
            pairs_per_row[j].append({"aspect": a, "sentiment": lab})

            aspect_stats_global[a][lab] += 1
            aspect_stats_global[a]["total"] += 1

        all_sentiments_sorted.extend(doc_sents)
        all_aspect_pairs_sorted.extend(pairs_per_row)

    all_sentiments = [None] * n
    all_aspect_pairs = [None] * n
    for orig_idx, sorted_idx in enumerate(sorted_indices):
        all_sentiments[sorted_idx] = all_sentiments_sorted[orig_idx]
        all_aspect_pairs[sorted_idx] = all_aspect_pairs_sorted[orig_idx]

    df["predicted_sentiment"] = all_sentiments
    df["predicted_aspects"] = all_aspect_pairs

    df.to_parquet(output_path, index=False, engine="pyarrow")
    print("Saved ->", output_path)

    del df, texts, sorted_texts, all_sentiments, all_aspect_pairs
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


Found 124 raw parts
Output Directory: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/Comments_ABSA

Skipping existing file: part_0000.parquet
Skipping existing file: part_0001.parquet
Skipping existing file: part_0002.parquet
Skipping existing file: part_0003.parquet
Skipping existing file: part_0004.parquet
Skipping existing file: part_0005.parquet
Skipping existing file: part_0006.parquet
Skipping existing file: part_0007.parquet
Skipping existing file: part_0008.parquet
Skipping existing file: part_0009.parquet
Skipping existing file: part_0010.parquet
Skipping existing file: part_0011.parquet
Skipping existing file: part_0012.parquet
Skipping existing file: part_0013.parquet
Skipping existing file: part_0014.parquet
Skipping existing file: part_0015.parquet
Skipping existing file: part_0016.parquet
Skipping existing file: part_0017.parquet
Skipping existing file: part_0018.parquet
Skipping existing file: part_0019.parquet
Skipping existing file: part_0020.parquet
Skipping e

part_0123.parquet:   0%|          | 0/12 [00:00<?, ?it/s]

/tmp/ipykernel_470/306367622.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):
/tmp/ipykernel_470/306367622.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Saved -> /content/drive/MyDrive/Colab Notebooks/Cleaned_output/Comments_ABSA/part_0123.parquet


#### Generate Aspect Statistics Report

In [6]:
print("\n" + "=" * 50)
print("GENERATING ASPECT PERCENTAGE REPORT...")
print("=" * 50)

report_data = []
for asp, counts in aspect_stats_global.items():
    tot = counts["total"]
    if tot >= 5:
        pos_pct = round((counts["positive"] / tot) * 100, 2)
        neg_pct = round((counts["negative"] / tot) * 100, 2)
        neu_pct = round((counts["neutral"] / tot) * 100, 2)
        report_data.append({
            "aspect": asp,
            "count": tot,
            "positive_pct": pos_pct,
            "negative_pct": neg_pct,
            "neutral_pct": neu_pct,
        })

df_report = pd.DataFrame(report_data).sort_values(by="count", ascending=False)

report_path = OUTPUT_DIR / "aspect_sentiment_percentages.parquet"
df_report.to_parquet(report_path, index=False, engine="pyarrow")

print(f"\nReport successfully saved to: {report_path}")
print("\nTop 20 Most Frequent Aspects & Their Sentiment Percentages:")
print(df_report.head(20).to_string(index=False))
print("\nAll tasks completed successfully!")


GENERATING ASPECT PERCENTAGE REPORT...

Report successfully saved to: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/Comments_ABSA/aspect_sentiment_percentages.parquet

Top 20 Most Frequent Aspects & Their Sentiment Percentages:
   aspect  count  positive_pct  negative_pct  neutral_pct
    کیفیت 302984         76.49         13.97         9.54
      جنس 178779         73.40         18.36         8.25
     جنسش 143657         60.92         29.94         9.14
     قیمت 132557         88.96          8.21         2.83
     کتاب 131407         89.93          3.31         6.76
    قیمتش  82323         65.70         31.40         2.91
بسته بندی  72248         78.12         19.98         1.90
   کیفیتش  65625         74.49         14.40        11.12
 ماندگاری  56364         49.45         33.99        16.56
دیجی کالا  54690         81.38         12.43         6.19
      بوی  52874         75.32         20.90         3.78
     رنگش  40678         49.51         45.28         5.21
      رنگ